In [1]:
import pandas as pd
import numpy as np

In [ ]:
csv_PATH = "부산항만공사_물동량 예측_20250331.csv"   #csv파일 
df = pd.read_csv(csv_PATH, encoding = 'cp949')   #csv  읽어오기





In [114]:
print(df.columns.tolist())     #컬럼 확인




['연도', '월', '연월', '수출입구분', '수출입구분설명', '물동량']


In [140]:
df_2425 = df.copy()   #파생변수 만들기 


In [141]:
df_2425 = df[(df['연도'] >= 2024) & (df['연도'] <= 2025)]   #csv 파일 중 '연도'컬럼 24년이상 25년이하로만 설정

df_2425 = df_2425.sort_values(                             #.sort_values('연도','월')로 오름차순 설정 
    by=['연도', '월'],
    ascending=[True, True]
)




In [ ]:
df.isna().sum() #결측값 확인  

연도         0
월          0
연월         0
수출입구분      0
수출입구분설명    0
물동량        0
dtype: int64

In [117]:
df_2425['분기'] = pd.cut(                                 #.cut() :  연속형 수치 데이터를 특정 구간(Bin)으로 나누어 범주형(Categorical) 데이터로 변환하는 함수
    df_2425['월'],                                        #'월'을 0~3 구간 3~6 6~9 9~12로 구간화 후 각각 1분기 ~ 4분기로 설정
    bins=[0, 3, 6, 9, 12],                                      
    labels=['1분기', '2분기', '3분기', '4분기']
)

df_2425 = df_2425.sort_values(                          #'연도','월'을 오름차순으로 설정 
    by=['연도', '월'],
    ascending=[True, True]
)




In [122]:
df_group = df_2425.groupby(                               #특정 열(Column)의 값이 같은 행(Row)끼리 하나의 그룹으로 묶어 요약 정보를 계산할 때 사용하는 명령어
    ['연도', '분기', '수출입구분설명']                         #'연도' ,'분기', '수출입구분설명'으로 묶기
) ['물동량'].sum().reset_index()                             #물동량 그룹별로 합계를 구한 뒤, 인덱스로 들어간 그룹 라벨을 다시 일반 열(Column)로 변환하는 코드

df_group

,연도,분기,수출입구분설명,물동량
0,2024,1분기,수입,1352857.50
1,2024,1분기,수출,1366166.75
2,2024,1분기,수입환적,1656608.50
3,2024,1분기,수출환적,1639101.00
4,2024,2분기,수입,1408204.00
5,2024,2분기,수출,1416705.75
6,2024,2분기,수입환적,1722986.00
7,2024,2분기,수출환적,1687279.00
8,2024,3분기,수입,1333555.75
9,2024,3분기,수출,1356115.75


In [126]:
순서 = ['수입', '수출', '수입환적', '수출환적'] 

df_2425['수출입구분설명'] = pd.Categorical( 
    df_2425['수출입구분설명'],
    categories=순서,
    ordered=True
)

df_2425

,연도,월,연월,수출입구분,수출입구분설명,물동량,분기
769,2024,1,2024-01,II,수입,428462.25,1분기
936,2024,1,2024-01,IT,수입환적,558618.00,1분기
1113,2024,1,2024-01,OT,수출환적,547360.25,1분기
1189,2024,1,2024-01,OO,수출,457956.00,1분기
234,2024,2,2024-02,IT,수입환적,525700.25,1분기
688,2024,2,2024-02,OO,수출,422256.25,1분기
1097,2024,2,2024-02,OT,수출환적,530238.00,1분기
1162,2024,2,2024-02,II,수입,401077.00,1분기
343,2024,3,2024-03,II,수입,523318.25,1분기
430,2024,3,2024-03,IT,수입환적,572290.25,1분기


In [130]:
df_2024 = df_group[
    (df_group['연도'] == 2024) &
    (df_group['분기'].isin(['1분기', '2분기', '3분기', '4분기']))
].copy()

기준값 = df_2024[
    df_2024['분기'] == '1분기'
][['수출입구분설명', '물동량']].rename(
    columns={'물동량': '1분기_기준물동량'}
)

df_2024 = df_2024.merge(
    기준값,
    on='수출입구분설명',
    how='left'
)

df_2024['증감률'] = (
    (df_2024['물동량'] - df_2024['1분기_기준물동량'])
    / df_2024['1분기_기준물동량']
    * 100 ).round(2).astype(str) + '%'


df_2024

,연도,분기,수출입구분설명,물동량,1분기_기준물동량,증감률
0,2024,1분기,수입,1352857.50,1352857.50,0.0%
1,2024,1분기,수출,1366166.75,1366166.75,0.0%
2,2024,1분기,수입환적,1656608.50,1656608.50,0.0%
3,2024,1분기,수출환적,1639101.00,1639101.00,0.0%
4,2024,2분기,수입,1408204.00,1352857.50,4.09%
5,2024,2분기,수출,1416705.75,1366166.75,3.7%
6,2024,2분기,수입환적,1722986.00,1656608.50,4.01%
7,2024,2분기,수출환적,1687279.00,1639101.00,2.94%
8,2024,3분기,수입,1333555.75,1352857.50,-1.43%
9,2024,3분기,수출,1356115.75,1366166.75,-0.74%


In [105]:
df_2024 = df_2024[
    ['연도', '분기', '수출입구분설명', '물동량', '증감률']
]

df_2024

,연도,분기,수출입구분설명,물동량,증감률
0,2024,1분기,수입,1352857.50,0.0%
1,2024,1분기,수출,1366166.75,0.0%
2,2024,1분기,수입환적,1656608.50,0.0%
3,2024,1분기,수출환적,1639101.00,0.0%
4,2024,2분기,수입,1408204.00,4.09%
5,2024,2분기,수출,1416705.75,3.7%
6,2024,2분기,수입환적,1722986.00,4.01%
7,2024,2분기,수출환적,1687279.00,2.94%
8,2024,3분기,수입,1333555.75,-1.43%
9,2024,3분기,수출,1356115.75,-0.74%


In [152]:
df_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   연도         16 non-null     int64   
 1   분기         16 non-null     category
 2   수출입구분설명    16 non-null     str     
 3   물동량        16 non-null     float64 
 4   1분기_기준물동량  16 non-null     float64 
 5   증감률        16 non-null     str     
dtypes: category(1), float64(2), int64(1), str(2)
memory usage: 992.0 bytes


In [154]:
df_group = df_2425.groupby(
    ['연도', '분기', '수출입구분설명']
)['물동량'].sum().reset_index()


In [155]:
순서 = ['수입', '수출', '수입환적', '수출환적']

df_2425['수출입구분설명'] = pd.Categorical(
    df_2425['수출입구분설명'],
    categories=순서,
    ordered=True
)

df_group = df_2425.groupby(
    ['연도', '분기', '수출입구분설명'],
    observed=True
)['물동량'].sum().reset_index()

df_group

,연도,분기,수출입구분설명,물동량
0,2023,1분기,수입,1308843.75
1,2023,1분기,수출,1324504.50
2,2023,1분기,수입환적,1525161.00
3,2023,1분기,수출환적,1516990.75
4,2023,2분기,수입,1375078.00
5,2023,2분기,수출,1397604.25
6,2023,2분기,수입환적,1557435.50
7,2023,2분기,수출환적,1586464.25
8,2023,3분기,수입,1319211.25
9,2023,3분기,수출,1339075.75


In [156]:
print(df_2425['수출입구분설명'].unique())

['수출환적', '수출', '수입', '수입환적']
Categories (4, str): ['수입' < '수출' < '수입환적' < '수출환적']


In [ ]:
dsasdsdasdasdasdaprint(df['수출입구분설명'].unique())

<StringArray>
['수출환적', '수입환적', '수출', '수입']
Length: 4, dtype: str


In [153]:
df_2425 = df.copy()

# 2023년 ~ 2024년 데이터만 설정
df_2425 = df[df['연도'].between(2023, 2024)]

# 월을 분기로 변환
df_2425['분기'] = pd.cut(
    df_2425['월'],
    bins=[0, 3, 6, 9, 12],
    labels=['1분기', '2분기', '3분기', '4분기']
)

# 연도 → 월 오름차순 정렬
df_2425 = df_2425.sort_values(
    by=['연도', '월'],
    ascending=[True, True]
)

# 연도, 분기, 수출입구분설명별 물동량 합계
df_group = df_2425.groupby(
    ['연도', '분기', '수출입구분설명']
)['물동량'].sum().reset_index()

# 수출입구분설명 순서 지정
순서 = ['수입', '수출', '수입환적', '수출환적']

df_group['수출입구분설명'] = pd.Categorical(
    df_group['수출입구분설명'],
    categories=순서,
    ordered=True
)

# 2023~2024년 전체
df_2023_2024 = df_group[
    (df_group['연도'].between(2023, 2024)) &
    (df_group['분기'].isin(['1분기', '2분기', '3분기', '4분기']))
].copy()

# 각 연도 + 수출입구분별 1분기 기준값
기준값 = df_2023_2024[
    df_2023_2024['분기'] == '1분기'
][['연도', '수출입구분설명', '물동량']].rename(
    columns={'물동량': '1분기_기준물동량'}
)

# 기준값 붙이기
df_2023_2024 = df_2023_2024.merge(
    기준값,
    on=['연도', '수출입구분설명'],
    how='left'
)

# 증감률 계산
df_2023_2024['증감률'] = (
    (df_2023_2024['물동량'] - df_2023_2024['1분기_기준물동량'])
    / df_2023_2024['1분기_기준물동량']
    * 100
).round(2).astype(str) + '%'

# 최종 결과
df_2023_2024 = df_2023_2024[
    ['연도', '분기', '수출입구분설명', '물동량', '증감률']
]

df_2023_2024

,연도,분기,수출입구분설명,물동량,증감률
0,2023,1분기,수입,1308843.75,0.0%
1,2023,1분기,수입환적,1525161.00,0.0%
2,2023,1분기,수출,1324504.50,0.0%
3,2023,1분기,수출환적,1516990.75,0.0%
4,2023,2분기,수입,1375078.00,5.06%
5,2023,2분기,수입환적,1557435.50,2.12%
6,2023,2분기,수출,1397604.25,5.52%
7,2023,2분기,수출환적,1586464.25,4.58%
8,2023,3분기,수입,1319211.25,0.79%
9,2023,3분기,수입환적,1553627.00,1.87%


# 24년도 비교 분석

- 1분기를 기준

2분기 수입 증감률 : 4.09% 증가 <br>
2분기 수출 증감률 : 3.7% 증기<br>
2분기 수출환적    : 2.94% 증가<br>
2분기 수입환적    : 4.01% 증가 <br>

3분기 수입       : 1.43% 감소<br>
3분기 수출       : 0.73% 감소<br>
3분기 수출환적    : 2.03% 증가<br>
3분기 수입환적    : 1.12% 증가 <br> 


4분기 수입      : 2.79% 감소<br>
4분기 수출      : 0.73% 감소<br>
4분기 수출환적   :3.94% 상승<br>
4분기 수입환적   :5.03% 상승<br>


In [142]:
df_23 = df.copy()   #파생변수 만들기 

df_23 = df[(df['연도'] == 2023)] #& (df['연도'] <= 2025)]   #csv 파일 중 '연도'컬럼 24년이상 25년이하로만 설정



df_23['분기'] = pd.cut(                                 #.cut() :  연속형 수치 데이터를 특정 구간(Bin)으로 나누어 범주형(Categorical) 데이터로 변환하는 함수
    df_23['월'],                                        #'월'을 0~3 구간 3~6 6~9 9~12로 구간화 후 각각 1분기 ~ 4분기로 설정
    bins=[0, 3, 6, 9, 12],                                      
    labels=['1분기', '2분기', '3분기', '4분기']
)

df_23 = df_23.sort_values(                          #'연도','월'을 오름차순으로 설정 
    by=['연도', '월'],
    ascending=[True, True]
)


df_group = df_23.groupby(                               #.group() : 특정 열(Column)의 값이 같은 행(Row)끼리 하나의 그룹으로 묶어 요약 정보를 계산할 때 사용하는 명령어
    ['연도', '분기', '수출입구분설명']                         #'연도' ,'분기', '수출입구분설명'으로 묶기
) ['물동량'].sum().reset_index()                            #'물동량'을 더하고 인덱스에 새로 저장 


순서 = ['수입', '수출', '수입환적', '수출환적'] 

df_23['수출입구분설명'] = pd.Categorical( 
    df_23['수출입구분설명'],
    categories=순서,
    ordered=True
)

df_2023 = df_group[                                         #24년도만 분석 
    (df_group['연도'] == 2023) &
    (df_group['분기'].isin(['1분기', '2분기', '3분기', '4분기']))
].copy()

기준값 = df_2023[
    df_2023['분기'] == '1분기'
][['수출입구분설명', '물동량']].rename(
    columns={'물동량': '1분기_기준물동량'}
)

df_2023 = df_2023.merge(
    기준값,
    on='수출입구분설명',
    how='left'
)

df_2023['증감률'] = (
    (df_2023['물동량'] - df_2023['1분기_기준물동량'])
    / df_2023['1분기_기준물동량']
    * 100 ).round(2).astype(str) + '%'


df_2023

,연도,분기,수출입구분설명,물동량,1분기_기준물동량,증감률
0,2023,1분기,수입,1308843.75,1308843.75,0.0%
1,2023,1분기,수입환적,1525161.00,1525161.00,0.0%
2,2023,1분기,수출,1324504.50,1324504.50,0.0%
3,2023,1분기,수출환적,1516990.75,1516990.75,0.0%
4,2023,2분기,수입,1375078.00,1308843.75,5.06%
5,2023,2분기,수입환적,1557435.50,1525161.00,2.12%
6,2023,2분기,수출,1397604.25,1324504.50,5.52%
7,2023,2분기,수출환적,1586464.25,1516990.75,4.58%
8,2023,3분기,수입,1319211.25,1308843.75,0.79%
9,2023,3분기,수입환적,1553627.00,1525161.00,1.87%


In [144]:
df_2325 = df[(df['연도'] >= 2023) & (df['연도'] <= 2025)]

df_2325 = df.copy()

# 2023년~2025년 데이터만 설정
df_2325 = df[
    (df['연도'] >= 2023) &
    (df['연도'] <= 2025)
]

# 2023~2025년 데이터 추출
df_2325 = df[
    (df['연도'] >= 2023) &
    (df['연도'] <= 2025)
].copy()


# 월을 분기로 변환
df_2325['분기'] = pd.cut(
    df_2325['월'],
    bins=[0, 3, 6, 9, 12],
    labels=['1분기', '2분기', '3분기', '4분기']
)


# 연도 → 월 오름차순 정렬
df_2325 = df_2325.sort_values(
    by=['연도', '월'],
    ascending=[True, True]
)


# 연도 + 분기 + 수출입구분별 물동량 합계
df_group = df_2325.groupby(
    ['연도', '분기', '수출입구분설명']
)['물동량'].sum().reset_index()


# 수입 → 수출 → 수입환적 → 수출환적 순서
순서 = ['수입', '수출', '수입환적', '수출환적']

df_group['수출입구분설명'] = pd.Categorical(
    df_group['수출입구분설명'],
    categories=순서,
    ordered=True
)

df_2023 = df_group[
    (df_group['연도'] == 2023) &
    (df_group['분기'].isin(['1분기', '2분기', '3분기', '4분기']))
].copy()


기준값 = df_2023[
    df_2023['분기'] == '1분기'
][['수출입구분설명', '물동량']].rename(
    columns={'물동량': '1분기_기준물동량'}
)


df_2023 = df_2023.merge(
    기준값,
    on='수출입구분설명',
    how='left'
)


df_2023['증감률'] = (
    (df_2023['물동량'] - df_2023['1분기_기준물동량'])
    / df_2023['1분기_기준물동량']
    * 100
).round(2).astype(str) + '%'


# 필요한 컬럼만 남기기
df_2023 = df_2023[
    ['연도', '분기', '수출입구분설명', '물동량', '증감률']
]

df_2023


df_2023_2024 = pd.concat(
    [df_2023, df_2024],
    ignore_index=True
)

df_2023_2024

,연도,분기,수출입구분설명,물동량,증감률,1분기_기준물동량
0,2023,1분기,수입,1308843.75,0.0%,NaN
1,2023,1분기,수입환적,1525161.00,0.0%,NaN
2,2023,1분기,수출,1324504.50,0.0%,NaN
3,2023,1분기,수출환적,1516990.75,0.0%,NaN
4,2023,2분기,수입,1375078.00,5.06%,NaN
5,2023,2분기,수입환적,1557435.50,2.12%,NaN
6,2023,2분기,수출,1397604.25,5.52%,NaN
7,2023,2분기,수출환적,1586464.25,4.58%,NaN
8,2023,3분기,수입,1319211.25,0.79%,NaN
9,2023,3분기,수입환적,1553627.00,1.87%,NaN


In [146]:
df_23 = df.copy()

# 2023년 데이터만 추출
df_23 = df[
    df['연도'] == 2023
].copy()


# 연도 → 월 순서대로 정렬
df_23 = df_23.sort_values(
    by=['연도', '월'],
    ascending=[True, True]
)


# 수입 → 수출 → 수입환적 → 수출환적 순서
순서 = ['수입', '수출', '수입환적', '수출환적']

df_23['수출입구분설명'] = pd.Categorical(
    df_23['수출입구분설명'],
    categories=순서,
    ordered=True
)


# 연도 + 월 + 수출입구분별 물동량 합계
df_group = df_23.groupby(
    ['연도', '월', '수출입구분설명']
)['물동량'].sum().reset_index()


# 1월 물동량을 기준값으로 가져오기
기준값 = df_group[
    df_group['월'] == 1
][['수출입구분설명', '물동량']].rename(
    columns={'물동량': '1월_기준물동량'}
)


# 각 수출입 구분에 1월 기준값 연결
df_group = df_group.merge(
    기준값,
    on='수출입구분설명',
    how='left'
)


# 1월 대비 증감률 계산
df_group['증감률'] = (
    (df_group['물동량'] - df_group['1월_기준물동량'])
    / df_group['1월_기준물동량']
    * 100
).round(2).astype(str) + '%'

df_2023 = df_group[
    ['연도', '월', '수출입구분설명', '물동량', '증감률']
]

df_2023

,연도,월,수출입구분설명,물동량,증감률
0,2023,1,수입,401408.00,0.0%
1,2023,1,수출,419471.25,0.0%
2,2023,1,수입환적,525101.00,0.0%
3,2023,1,수출환적,493326.25,0.0%
4,2023,2,수입,410605.50,2.29%
5,2023,2,수출,433076.00,3.24%
6,2023,2,수입환적,442852.00,-15.66%
7,2023,2,수출환적,472133.00,-4.3%
8,2023,3,수입,496830.25,23.77%
9,2023,3,수출,471957.25,12.51%


In [149]:
# 1. 2023년 월별 전체 물동량 합계 집계

newdf_2023 = df[df['연도'] == 2023].groupby(
    '월',
    as_index=False
)['물동량'].sum()


# 2. 2023년 연간 월평균 물동량 산출

annual_monthly_mean_2023 = newdf_2023['물동량'].mean()


# 3. 월평균 대비 편차(계절성 지수, %) 계산

newdf_2023['평균대비(%)'] = (
    (newdf_2023['물동량'] - annual_monthly_mean_2023)
    / annual_monthly_mean_2023
    * 100
)


# 결과 확인

newdf_2023

,월,물동량,평균대비(%)
0,1,1839306.50,-4.672424
1,2,1758666.50,-8.851834
2,3,2077527.00,7.674068
3,4,2024480.50,4.924774
4,5,1989936.25,3.134415
5,6,1902165.25,-1.414580
6,7,1896403.75,-1.713187
7,8,1893300.75,-1.874010
8,9,1965505.50,1.868218
9,10,1895968.25,-1.735759


In [151]:
# 1. 2024년 월별 전체 물동량 합계 집계
newdf_2024 = df[df['연도'] == 2024].groupby(
    '월',
    as_index=False
)['물동량'].sum()

# 2. 2024년 연간 월평균 물동량 산출
annual_monthly_mean_2024 = newdf_2024['물동량'].mean()

# 3. 월평균 대비 편차(계절성 지수, %) 계산
newdf_2024['평균대비(%)'] = (
    (newdf_2024['물동량'] - annual_monthly_mean_2024)
    / annual_monthly_mean_2024
    * 100
)

newdf_2024

,월,물동량,평균대비(%)
0,1,1992396.50,-2.021398
1,2,1879271.50,-7.584462
2,3,2143065.75,5.387951
3,4,2046147.00,0.621850
4,5,2098141.00,3.178720
5,6,2090886.75,2.821984
6,7,2107241.75,3.626261
7,8,2052806.00,0.949315
8,9,1877152.00,-7.688691
9,10,2049390.75,0.781366
